In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

BASE = "/content/drive/MyDrive/hate_speech_ft"
os.makedirs(BASE, exist_ok=True)
os.makedirs(f"{BASE}/outputs", exist_ok=True)
os.makedirs(f"{BASE}/hf_cache", exist_ok=True)

os.environ["HF_HOME"] = f"{BASE}/hf_cache"
os.environ["HF_HUB_CACHE"] = f"{BASE}/hf_cache/hub"

In [ ]:
REPO_URL = "https://github.com/Naaeeen/hate-speech-ft.git"
REPO_DIR = "/content/hate-speech-ft"

!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR

Cloning into '/content/hate-speech-ft'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 24 (delta 4), reused 22 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 8.62 KiB | 8.63 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/hate-speech-ft


In [ ]:
%cd /content/hate-speech-ft
!git pull

/content/hate-speech-ft
Already up to date.


In [ ]:
CACHE_DIR = "/content/drive/MyDrive/hate-speech-ft/pip_cache"
!pip install -q -r requirements-colab.txt --cache-dir $CACHE_DIR

In [ ]:
import torch
import transformers
import datasets
import peft

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu130
transformers: 5.6.0
datasets: 3.6.0
peft: 0.19.1
cuda available: False


In [ ]:
!python src/utils/env_check.py
!python src/data/load_hatexplain.py
!python src/data/check_dataset.py
!python src/models/load_distilbert.py
!python src/models/model_forward_check.py
!python src/models/peft_check.py

Environment check
----------------------------------------
Torch version: 2.11.0+cu130
CUDA available: False
Transformers version: 5.6.0
Datasets version: 3.6.0
PEFT version: 0.19.1
HF_HOME: /content/drive/MyDrive/hate_speech_ft/hf_cache
HF_HUB_CACHE: /content/drive/MyDrive/hate_speech_ft/hf_cache/hub
DatasetDict({
    train: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1924
    })
})

Train sample:
{'id': '23107796_gab', 'annotators': {'label': [0, 2, 2], 'annotator_id': [203, 204, 233], 'target': [['Hindu', 'Islam'], ['Hindu', 'Islam'], ['Hindu', 'Islam', 'Other']]}, 'rationales': [[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

texts = [
    "I love this!",
    "This is terrible."
]

results = classifier(texts)
print(results)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998764991760254}, {'label': 'NEGATIVE', 'score': 0.9996345043182373}]


In [ ]:
!python src/run_distilbert_hatexplain.py \
  --max_train_samples 64 \
  --max_eval_samples 64 \
  --num_train_epochs 1 \
  --output_dir outputs/distilbert_hatexplain_smoke